In [1]:
"""
CRYPTO INDICATOR PROCESSOR
===========================
Reads crypto_top500_2yr.csv (or any per-coin CSVs in crypto_data/),
calculates all technical indicators (same as stock processor),
and saves to crypto_with_indicators.csv

Input columns : date, open, high, low, close, volume_usd, symbol, ...
Output        : all input cols + 50+ indicators + surge_score
"""

import pandas as pd
import numpy as np
import warnings
import time
import os
warnings.filterwarnings('ignore')


# ============================================================================
# SURGE SCORE HELPERS
# ============================================================================

MOMENTUM_INDICATORS = ['RSI', 'Stoch_K', 'Stoch_D', 'Williams_R', 'CCI', 'ROC_14', 'TSI']
SURGE_LOOKBACK  = 3
SURGE_THRESHOLD = 10

def _normalize_indicator(series, window=252):
    series = series.fillna(series.median())
    rolling_min = series.rolling(window, min_periods=20).min()
    rolling_max = series.rolling(window, min_periods=20).max()
    return ((series - rolling_min) / (rolling_max - rolling_min)).clip(0, 1).mul(100).fillna(50)

def _get_momentum_score(df):
    vals, weights = [], []
    for col in MOMENTUM_INDICATORS:
        if col in df.columns:
            vals.append(_normalize_indicator(df[col]))
            weights.append(1)
    if vals:
        return sum(vals) / sum(weights)
    return pd.Series(50.0, index=df.index)

def _compute_surge_score(momentum_series, lookback=SURGE_LOOKBACK):
    abs_chg = momentum_series.diff(lookback)
    pct_chg = momentum_series.pct_change(lookback) * 100
    accel   = momentum_series.diff().diff()
    mean    = momentum_series.rolling(20, min_periods=5).mean()
    std     = momentum_series.rolling(20, min_periods=5).std()
    z       = (momentum_series - mean) / (std + 1e-10)

    score = pd.Series(0.0, index=momentum_series.index)
    for i in range(lookback, len(momentum_series)):
        score.iloc[i] = (
            abs_chg.iloc[i] * 0.4 +
            (pct_chg.iloc[i] if not pd.isna(pct_chg.iloc[i]) else 0) * 0.3 +
            (accel.iloc[i]   if not pd.isna(accel.iloc[i])   else 0) * 2.0 +
            (z.iloc[i]       if not pd.isna(z.iloc[i])        else 0) * 3.0
        )
    return score


# ============================================================================
# TECHNICAL INDICATORS
# ============================================================================

class TechnicalIndicators:

    @staticmethod
    def calculate_all(df):
        ind = TechnicalIndicators

        # ── Trend ──────────────────────────────────────────────────────────────
        df['EMA_9']   = df['close'].ewm(span=9,   adjust=False).mean()
        df['EMA_21']  = df['close'].ewm(span=21,  adjust=False).mean()
        df['EMA_50']  = df['close'].ewm(span=50,  adjust=False).mean()
        df['EMA_100'] = df['close'].ewm(span=100, adjust=False).mean()
        df['EMA_200'] = df['close'].ewm(span=200, adjust=False).mean()
        df['SMA_50']  = df['close'].rolling(50).mean()
        df['SMA_200'] = df['close'].rolling(200).mean()

        # ── Bollinger Bands ────────────────────────────────────────────────────
        df['BB_Middle']  = df['close'].rolling(20).mean()
        bb_std           = df['close'].rolling(20).std()
        df['BB_Upper']   = df['BB_Middle'] + bb_std * 2
        df['BB_Lower']   = df['BB_Middle'] - bb_std * 2
        df['BB_Width']   = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Middle'].replace(0, np.nan)
        df['BB_Percent'] = (df['close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower']).replace(0, np.nan)

        # ── Keltner Channels ───────────────────────────────────────────────────
        df['KC_Middle'] = df['close'].ewm(span=20, adjust=False).mean()
        atr_kc          = ind.calculate_atr(df, 20)
        df['KC_Upper']  = df['KC_Middle'] + atr_kc * 2
        df['KC_Lower']  = df['KC_Middle'] - atr_kc * 2

        # ── Donchian Channels ──────────────────────────────────────────────────
        df['DC_Upper']  = df['high'].rolling(20).max()
        df['DC_Lower']  = df['low'].rolling(20).min()
        df['DC_Middle'] = (df['DC_Upper'] + df['DC_Lower']) / 2

        # ── Pivot Points ───────────────────────────────────────────────────────
        df['Pivot'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
        df['R1']    = 2 * df['Pivot'] - df['low'].shift(1)
        df['S1']    = 2 * df['Pivot'] - df['high'].shift(1)
        df['R2']    = df['Pivot'] + (df['high'].shift(1) - df['low'].shift(1))
        df['S2']    = df['Pivot'] - (df['high'].shift(1) - df['low'].shift(1))

        # ── VWAP ───────────────────────────────────────────────────────────────
        df['VWAP'] = (
            df['volume_usd'] * (df['high'] + df['low'] + df['close']) / 3
        ).cumsum() / df['volume_usd'].cumsum()

        # ── Supertrend ─────────────────────────────────────────────────────────
        st = ind.calculate_supertrend(df)
        df['Supertrend']           = st['Supertrend']
        df['Supertrend_Direction'] = st['Direction']

        # ── Parabolic SAR ──────────────────────────────────────────────────────
        psar = ind.calculate_psar(df)
        df['PSAR']       = psar['PSAR']
        df['PSAR_Trend'] = psar['Trend']

        # ── Ichimoku ───────────────────────────────────────────────────────────
        for k, v in ind.calculate_ichimoku(df).items():
            df[k] = v

        # ── Momentum ───────────────────────────────────────────────────────────
        df['RSI']        = ind.calculate_rsi(df['close'], 14)
        stoch            = ind.calculate_stochastic(df)
        df['Stoch_K']    = stoch['K']
        df['Stoch_D']    = stoch['D']
        df['Williams_R'] = ind.calculate_williams_r(df)
        df['CCI']        = ind.calculate_cci(df)
        df['ROC_9']      = ((df['close'] - df['close'].shift(9))  / df['close'].shift(9).replace(0, np.nan))  * 100
        df['ROC_14']     = ((df['close'] - df['close'].shift(14)) / df['close'].shift(14).replace(0, np.nan)) * 100
        df['ROC_21']     = ((df['close'] - df['close'].shift(21)) / df['close'].shift(21).replace(0, np.nan)) * 100

        # ── MACD ───────────────────────────────────────────────────────────────
        macd             = ind.calculate_macd(df['close'])
        df['MACD']       = macd['MACD']
        df['MACD_Signal']= macd['Signal']
        df['MACD_Hist']  = macd['Histogram']

        # ── TSI ────────────────────────────────────────────────────────────────
        df['TSI'] = ind.calculate_tsi(df['close'])

        # ── Volume Indicators ──────────────────────────────────────────────────
        df['OBV']        = ind.calculate_obv(df)
        df['AD_Line']    = ind.calculate_ad_line(df)
        df['CMF']        = ind.calculate_cmf(df)
        df['MFI']        = ind.calculate_mfi(df)
        df['Volume_ROC'] = ((df['volume_usd'] - df['volume_usd'].shift(14)) / df['volume_usd'].shift(14).replace(0, np.nan)) * 100

        # ── Volatility ─────────────────────────────────────────────────────────
        df['ATR'] = ind.calculate_atr(df, 14)

        # ── Directional Movement ───────────────────────────────────────────────
        dm             = ind.calculate_dm(df)
        df['PLUS_DI']  = dm['PLUS_DI']
        df['MINUS_DI'] = dm['MINUS_DI']
        df['ADX']      = dm['ADX']

        # ── Aroon ──────────────────────────────────────────────────────────────
        aroon                  = ind.calculate_aroon(df)
        df['Aroon_Up']         = aroon['Aroon_Up']
        df['Aroon_Down']       = aroon['Aroon_Down']
        df['Aroon_Oscillator'] = aroon['Aroon_Oscillator']

        # ── Price Structure ────────────────────────────────────────────────────
        df['Higher_Highs'] = (df['high'] > df['high'].shift(1)).astype(int)
        df['Higher_Lows']  = (df['low']  > df['low'].shift(1)).astype(int)

        # ── Future Returns (for backtesting / ML) ─────────────────────────────
        df['Next_1d_Return'] = ((df['close'].shift(-1) - df['close']) / df['close']) * 100
        df['Next_2d_Return'] = ((df['close'].shift(-2) - df['close']) / df['close']) * 100
        df['Next_3d_Return'] = ((df['close'].shift(-3) - df['close']) / df['close']) * 100
        df['Next_5d_Return'] = ((df['close'].shift(-5) - df['close']) / df['close']) * 100
        df['Next_7d_Return'] = ((df['close'].shift(-7) - df['close']) / df['close']) * 100

        # ── Surge Score ────────────────────────────────────────────────────────
        momentum_composite = _get_momentum_score(df)
        df['surge_score']  = _compute_surge_score(momentum_composite, lookback=SURGE_LOOKBACK)

        return df

    # ── Individual indicator methods ──────────────────────────────────────────

    @staticmethod
    def calculate_atr(df, period=14):
        hl        = df['high'] - df['low']
        hc        = np.abs(df['high'] - df['close'].shift())
        lc        = np.abs(df['low']  - df['close'].shift())
        tr        = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        return tr.rolling(period).mean()

    @staticmethod
    def calculate_supertrend(df, period=10, multiplier=3):
        atr    = TechnicalIndicators.calculate_atr(df, period)
        hl_avg = (df['high'] + df['low']) / 2
        upper  = hl_avg + multiplier * atr
        lower  = hl_avg - multiplier * atr

        supertrend = pd.Series(index=df.index, dtype=float)
        direction  = pd.Series(index=df.index, dtype=int)
        supertrend.iloc[0] = lower.iloc[0]
        direction.iloc[0]  = 1

        for i in range(1, len(df)):
            if direction.iloc[i-1] == 1:
                cur_lower = max(lower.iloc[i], supertrend.iloc[i-1])
            else:
                cur_lower = lower.iloc[i]
            if direction.iloc[i-1] == -1:
                cur_upper = min(upper.iloc[i], supertrend.iloc[i-1])
            else:
                cur_upper = upper.iloc[i]

            if df['close'].iloc[i] > supertrend.iloc[i-1]:
                supertrend.iloc[i] = cur_lower
                direction.iloc[i]  = 1
            else:
                supertrend.iloc[i] = cur_upper
                direction.iloc[i]  = -1

        return {'Supertrend': supertrend, 'Direction': direction}

    @staticmethod
    def calculate_psar(df, iaf=0.02, maxaf=0.2):
        psar  = df['close'].copy()
        trend = pd.Series(index=df.index, dtype=int)
        bull  = True
        af    = iaf
        hp    = df['high'].iloc[0]
        lp    = df['low'].iloc[0]
        trend.iloc[0] = 1

        for i in range(2, len(df)):
            if bull:
                psar.iloc[i] = psar.iloc[i-1] + af * (hp - psar.iloc[i-1])
            else:
                psar.iloc[i] = psar.iloc[i-1] + af * (lp - psar.iloc[i-1])

            reverse = False
            if bull:
                if df['low'].iloc[i] < psar.iloc[i]:
                    bull = False; reverse = True
                    psar.iloc[i] = hp; lp = df['low'].iloc[i]; af = iaf
            else:
                if df['high'].iloc[i] > psar.iloc[i]:
                    bull = True; reverse = True
                    psar.iloc[i] = lp; hp = df['high'].iloc[i]; af = iaf

            if not reverse:
                if bull:
                    if df['high'].iloc[i] > hp:
                        hp = df['high'].iloc[i]; af = min(af + iaf, maxaf)
                    if df['low'].iloc[i-1] < psar.iloc[i]: psar.iloc[i] = df['low'].iloc[i-1]
                    if df['low'].iloc[i-2] < psar.iloc[i]: psar.iloc[i] = df['low'].iloc[i-2]
                else:
                    if df['low'].iloc[i] < lp:
                        lp = df['low'].iloc[i]; af = min(af + iaf, maxaf)
                    if df['high'].iloc[i-1] > psar.iloc[i]: psar.iloc[i] = df['high'].iloc[i-1]
                    if df['high'].iloc[i-2] > psar.iloc[i]: psar.iloc[i] = df['high'].iloc[i-2]

            trend.iloc[i] = 1 if bull else -1

        return {'PSAR': psar, 'Trend': trend}

    @staticmethod
    def calculate_ichimoku(df):
        t = (df['high'].rolling(9).max()  + df['low'].rolling(9).min())  / 2
        k = (df['high'].rolling(26).max() + df['low'].rolling(26).min()) / 2
        return {
            'Ichimoku_Tenkan': t,
            'Ichimoku_Kijun':  k,
            'Ichimoku_SpanA':  ((t + k) / 2).shift(26),
            'Ichimoku_SpanB':  ((df['high'].rolling(52).max() + df['low'].rolling(52).min()) / 2).shift(26),
        }

    @staticmethod
    def calculate_rsi(series, period=14):
        delta = series.diff()
        gain  = delta.where(delta > 0, 0).rolling(period).mean()
        loss  = (-delta.where(delta < 0, 0)).rolling(period).mean()
        return 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    @staticmethod
    def calculate_stochastic(df, k=14, d=3):
        lo = df['low'].rolling(k).min()
        hi = df['high'].rolling(k).max()
        K  = 100 * ((df['close'] - lo) / (hi - lo).replace(0, np.nan))
        return {'K': K, 'D': K.rolling(d).mean()}

    @staticmethod
    def calculate_williams_r(df, period=14):
        hi = df['high'].rolling(period).max()
        lo = df['low'].rolling(period).min()
        return -100 * ((hi - df['close']) / (hi - lo).replace(0, np.nan))

    @staticmethod
    def calculate_cci(df, period=20):
        tp  = (df['high'] + df['low'] + df['close']) / 3
        sma = tp.rolling(period).mean()
        mad = tp.rolling(period).apply(lambda x: np.abs(x - x.mean()).mean())
        return (tp - sma) / (0.015 * mad).replace(0, np.nan)

    @staticmethod
    def calculate_macd(series, fast=12, slow=26, signal=9):
        m = series.ewm(span=fast, adjust=False).mean() - series.ewm(span=slow, adjust=False).mean()
        s = m.ewm(span=signal, adjust=False).mean()
        return {'MACD': m, 'Signal': s, 'Histogram': m - s}

    @staticmethod
    def calculate_tsi(series, long=25, short=13):
        pc  = series.diff()
        ds  = pc.ewm(span=long, adjust=False).mean().ewm(span=short, adjust=False).mean()
        dsa = pc.abs().ewm(span=long, adjust=False).mean().ewm(span=short, adjust=False).mean()
        return 100 * (ds / dsa.replace(0, np.nan))

    @staticmethod
    def calculate_obv(df):
        obv = [0]
        for i in range(1, len(df)):
            if df['close'].iloc[i] > df['close'].iloc[i-1]:
                obv.append(obv[-1] + df['volume_usd'].iloc[i])
            elif df['close'].iloc[i] < df['close'].iloc[i-1]:
                obv.append(obv[-1] - df['volume_usd'].iloc[i])
            else:
                obv.append(obv[-1])
        return pd.Series(obv, index=df.index)

    @staticmethod
    def calculate_ad_line(df):
        clv = ((df['close'] - df['low']) - (df['high'] - df['close'])) / (df['high'] - df['low']).replace(0, np.nan)
        return (clv.fillna(0) * df['volume_usd']).cumsum()

    @staticmethod
    def calculate_cmf(df, period=20):
        mfm = ((df['close'] - df['low']) - (df['high'] - df['close'])) / (df['high'] - df['low']).replace(0, np.nan)
        mfv = mfm.fillna(0) * df['volume_usd']
        return mfv.rolling(period).sum() / df['volume_usd'].rolling(period).sum().replace(0, np.nan)

    @staticmethod
    def calculate_mfi(df, period=14):
        tp  = (df['high'] + df['low'] + df['close']) / 3
        rmf = tp * df['volume_usd']
        pos = pd.Series(0.0, index=df.index)
        neg = pd.Series(0.0, index=df.index)
        for i in range(1, len(df)):
            if   tp.iloc[i] > tp.iloc[i-1]: pos.iloc[i] = rmf.iloc[i]
            elif tp.iloc[i] < tp.iloc[i-1]: neg.iloc[i] = rmf.iloc[i]
        mfi = 100 - (100 / (1 + pos.rolling(period).sum() / neg.rolling(period).sum().replace(0, np.nan)))
        return mfi.fillna(50)

    @staticmethod
    def calculate_dm(df, period=14):
        hd = df['high'].diff()
        ld = -df['low'].diff()
        pdm = pd.Series(0.0, index=df.index)
        mdm = pd.Series(0.0, index=df.index)
        pdm[(hd > ld) & (hd > 0)] = hd
        mdm[(ld > hd) & (ld > 0)] = ld
        atr     = TechnicalIndicators.calculate_atr(df, period)
        pdi     = 100 * pdm.rolling(period).mean() / atr.replace(0, np.nan)
        mdi     = 100 * mdm.rolling(period).mean() / atr.replace(0, np.nan)
        dx      = 100 * np.abs(pdi - mdi) / (pdi + mdi).replace(0, np.nan)
        return {'PLUS_DI': pdi, 'MINUS_DI': mdi, 'ADX': dx.rolling(period).mean()}

    @staticmethod
    def calculate_aroon(df, period=25):
        def up(x):   return ((period - (len(x) - 1 - np.argmax(x))) / period) * 100
        def down(x): return ((period - (len(x) - 1 - np.argmin(x))) / period) * 100
        au = df['high'].rolling(period + 1).apply(up,   raw=True)
        ad = df['low'].rolling(period + 1).apply(down,  raw=True)
        return {'Aroon_Up': au, 'Aroon_Down': ad, 'Aroon_Oscillator': au - ad}


# ============================================================================
# MAIN PROCESSING
# ============================================================================

def process_crypto_data(input_csv: str, output_csv: str = "crypto_with_indicators.csv"):
    print("=" * 70)
    print("CRYPTO INDICATOR PROCESSOR")
    print("=" * 70)

    if not os.path.exists(input_csv):
        print(f"❌ File not found: {input_csv}")
        return None

    print(f"\n📂 Reading: {input_csv}")
    df = pd.read_csv(input_csv, parse_dates=["date"])
    df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

    # Drop old partial indicator columns (will be recomputed cleanly)
    old_indicator_cols = [
        "daily_return_pct","candle_range_pct","ma_20","ma_50","ma_200",
        "vol_ma_20","rel_volume","rsi_14","bb_upper","bb_lower","bb_pct",
        "macd","macd_signal","macd_hist","bb_mid","bb_std"
    ]
    df = df.drop(columns=[c for c in old_indicator_cols if c in df.columns])

    symbols   = df["symbol"].unique()
    print(f"✓ Loaded {len(df):,} rows | {len(symbols)} coins\n")

    all_data   = []
    successful = 0
    failed     = 0
    start      = time.time()

    print("=" * 70)
    for idx, symbol in enumerate(symbols, 1):
        try:
            coin_df = df[df["symbol"] == symbol].copy().reset_index(drop=True)

            if len(coin_df) < 50:
                print(f"[{idx:3d}/{len(symbols)}] ⚠  {symbol:12s} — only {len(coin_df)} rows, skipping (need 50+)")
                failed += 1
                continue

            coin_df = TechnicalIndicators.calculate_all(coin_df)
            all_data.append(coin_df)
            successful += 1

            latest = coin_df.iloc[-1]
            print(
                f"[{idx:3d}/{len(symbols)}] ✓ {symbol:12s} | "
                f"{len(coin_df):3d} days | "
                f"RSI: {latest.get('RSI', float('nan')):.1f} | "
                f"surge: {latest.get('surge_score', float('nan')):.2f}"
            )

        except Exception as e:
            print(f"[{idx:3d}/{len(symbols)}] ❌ {symbol:12s} — {str(e)[:60]}")
            failed += 1

    print("=" * 70)

    if not all_data:
        print("❌ No data processed.")
        return None

    print(f"\n💾 Merging and saving...")
    out = pd.concat(all_data, ignore_index=True)
    out = out.sort_values(["symbol", "date"]).reset_index(drop=True)
    out.to_csv(output_csv, index=False)

    elapsed = time.time() - start

    # Summary
    basic  = ["date","open","high","low","close","volume_usd","symbol","name","coin_id"]
    meta   = ["market_cap_usd","cap_tier","ath","atl","circulating_supply","max_supply",
              "change_24h_pct","volume_24h_usd","is_stablecoin","pair"]
    future = [c for c in out.columns if c.startswith("Next_")]
    indic  = [c for c in out.columns if c not in basic + meta + future]

    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"✅ Successful : {successful} coins")
    print(f"❌ Failed     : {failed} coins")
    print(f"📊 Total rows : {len(out):,}")
    print(f"📅 Date range : {out['date'].min().date()} → {out['date'].max().date()}")
    print(f"⏱  Time       : {elapsed:.1f}s ({elapsed/60:.1f} min)")
    print(f"\nColumn breakdown:")
    print(f"  OHLCV + meta : {len(basic + meta)}")
    print(f"  Indicators   : {len(indic)}  ← includes surge_score")
    print(f"  Future returns: {len(future)}")
    print(f"  TOTAL        : {len(out.columns)}")
    print(f"\n📁 Output: {output_csv}")
    print(f"   Size  : {out.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

    return out


if __name__ == "__main__":
    INPUT_CSV  = "crypto_top500_2yr.csv"
    OUTPUT_CSV = "crypto_with_indicators.csv"

    result = process_crypto_data(INPUT_CSV, OUTPUT_CSV)

CRYPTO INDICATOR PROCESSOR

📂 Reading: crypto_top500_2yr.csv
✓ Loaded 123,310 rows | 213 coins

[  1/213] ✓ 0G           | 180 days | RSI: 29.3 | surge: -26.27
[  2/213] ✓ 1INCH        | 730 days | RSI: 52.2 | surge: -14.40
[  3/213] ✓ 2Z           | 170 days | RSI: 48.1 | surge: -34.58
[  4/213] ✓ A            | 297 days | RSI: 53.4 | surge: -29.57
[  5/213] ✓ AAVE         | 730 days | RSI: 47.7 | surge: -15.48
[  6/213] ✓ ADA          | 730 days | RSI: 54.4 | surge: -3.92
[  7/213] ✓ AEUR         | 730 days | RSI: 39.7 | surge: -19.58
[  8/213] ✓ ALGO         | 730 days | RSI: 57.9 | surge: -2.53
[  9/213] ✓ AMP          | 730 days | RSI: 26.0 | surge: -25.31
[ 10/213] ✓ ANKR         | 730 days | RSI: 61.0 | surge: 2.77
[ 11/213] ✓ APE          | 730 days | RSI: 40.8 | surge: -30.39
[ 12/213] ✓ APT          | 730 days | RSI: 54.6 | surge: 57.08
[ 13/213] ✓ AR           | 730 days | RSI: 53.5 | surge: -4.60
[ 14/213] ✓ ARB          | 730 days | RSI: 49.2 | surge: -4.80
[ 15/213] ✓ ARD